In [ ]:
import glob
import logging
import logging.config
import os

import pandas as pd

from graal.summary.blind_eval_project import BlindEvalProject

logging.config.fileConfig("logging.conf")

DATA_FOLDER = os.getenv("DATA_FOLDER", "data")
EXCEL_OUTPUT_FILE = "data/blind_eval_summaries/resultat_eval_aveugle.xlsx"
PROJECT_SAVE_LOCATION = "data/blind_eval_summaries/project_eval_aveugle_objets.pkl"
METRICS = ["Concision", "Fidélité au texte", "Formulation"]
OUTPUT_COLUMNS = ["ID", "Source"] + METRICS

result_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

project = BlindEvalProject.load_from_disk(PROJECT_SAVE_LOCATION)
logging.info(f"Project '{PROJECT_SAVE_LOCATION}' successfully loaded")
display(project.mapping_obj_to_author)


RESPONSES_FOLDER = f"{DATA_FOLDER}/blind_eval_summaries/réponses/"

# Get all Excel files in the RESPONSES_FOLDER
excel_files = glob.glob(os.path.join(RESPONSES_FOLDER, "*.xlsx"))

# Read each Excel file into a dataframe and store them in a list
dataframes = [pd.read_excel(file) for file in excel_files]

# Display the list of dataframes
for i, df in enumerate(dataframes):
    print(f"DataFrame {i} from file {excel_files[i]}:")
    for metric in METRICS:
        for col in [f"1 - {metric}", f"2 - {metric}"]:
            df[col] = df[col].apply(lambda x: 1 if x == "oui" else 0).astype(int)
    df["Objet 1"] = df["ID"].apply(
        lambda x: project.mapping_obj_to_author.get(x, {}).get("Objet 1", "")
    )
    df["Objet 2"] = df["ID"].apply(
        lambda x: project.mapping_obj_to_author.get(x, {}).get("Objet 2", "")
    )
    rows_to_append = []
    for index, row in df.iterrows():
        row_to_append_obj1 = {"ID": row["ID"], "Source": row["Objet 1"]}
        row_to_append_obj2 = {"ID": row["ID"], "Source": row["Objet 2"]}
        for metric in METRICS:
            row_to_append_obj1[metric] = row[f"1 - {metric}"]
            row_to_append_obj2[metric] = row[f"2 - {metric}"]
        rows_to_append.append(row_to_append_obj1)
        rows_to_append.append(row_to_append_obj2)
    result_df = pd.concat([result_df, pd.DataFrame(rows_to_append)], ignore_index=True)
    display(df)

display(result_df)
# Sum up the metrics columns of result_df grouped by "ID" and "Source"
summary_df = result_df.groupby(["ID", "Source"]).sum().reset_index()

# Display the summary dataframe
display(summary_df)

# Save the summary dataframe to an Excel file
summary_df.to_excel(EXCEL_OUTPUT_FILE, index=False)
logging.info(f"Summary saved to '{EXCEL_OUTPUT_FILE}'")

os.system(f'open "{EXCEL_OUTPUT_FILE}"')